# Problem Description

This report demonstrates 3 iterative numerical methods for solving matrix equation: Jordan, Gauss-Seidel and SOR method.

$$ Ax = b, A=(a_{ij})_{n\times n}\in\mathbb{R}^{n\times n}, b\in\mathbb{R}^n $$

**Algorithm**

**Jordan method** decomposite $A$ to a diagonal and a zero-diagonal remaining matrix, that is let

$$ D=\mathrm{diag}\{a_{11},a_{22},\cdots, a_{nn}\}, \text{ then } Ax = (D + (A - D)) x = b $$

if $D$ is invertible, then rearranging the equation, we have 

$$ x = D^{-1}(D-A)x + D^{-1}b, \text{ or rewritten in iteration form } x^{(k+1)}=(I-D^{-1}A)x^{(k)}+D^{-1}b=Nx^{(k)}+y$$

Algorithm description:

```text
initiate x1, x2 such that x2 != x1, n = 0
D = diag(A)
N = inverse(D) * (D-A), y = inverse(D) * b
while norm(x1 - x2) > tol and n < max_iteration
    x1 = x2
    x2 = N * x1 + y
    n = n + 1
end-while
```

**Gauss-Seidel** method takes variables that are already iterated into account, whereas decompose $A$ into 2 triangular matrices, that is 

$$ D=\mathrm{diag}\{a_{11},a_{22},\cdots, a_{nn}\}, \text{ then } Ax = ((L + D) + U) x = b $$

if $D$ is invertible, then rearranging the equation, we have 

$$ x = -(D+L)^{-1}Ux+(D+L)^{-1}b, \text{ or rewritten in iteration form } x^{(k+1)}=-(D+L)^{-1}Ux^{(k)}+(D+L)^{-1}b=Nx^{(k)}+y$$

Algorithm description:
```text
initiate x1, x2 such that x2 != x1, n = 0
D = diag(A)
L = lower(A)
N = -inverse(D + L) * U, y = inverse(D + L) * b
while norm(x1 - x2) > tol and n < max_iteration
    x1 = x2
    # x2 = N * x1 + y
    for u = 1 to n
        x1(u) = x2(u)
    end-for
    for i = 1 to n
        s = 0
        for j = 1 to n
            s = s + a(i, i) * x2(j)
        x2(i) = (b(i) - s + a(i, i) * x2(i)) / a(i, i)
    end-for
    n = n + 1
end-while
```

**SOR** method introduces constant slack weigh into Gauss-Seidel method. Rerite the iteration formula of Gauss-Seidel method into 

$$ x^{(k+1)}= -D^{-1}Lx^{(k+1)}-D^{-1}Ux^{(k)}+D^{-1}b $$

let $\tilde{L}=-D^{-1}L,\tilde{U}=-D^{-1}U, y = D^{-1}b$, it becomes

$$ x^{(k+1)}=\tilde{L}x^{(k+1)}+\tilde{U}x^{(k)}+y $$

Taking weighted $x^{(k)}$ and $x^{(k+1)}$ calculated by Gauss-Seidel method (weighted by $w$ and $1-w$), we have

$$
x^{(k+1)}=(1-w)x^{(k)}+wx^{(k+1)} = (1-w)x^{(k)} + w(\tilde{L} x^{(k+1)}+\tilde{U}x^{(k)}+y)
$$

Furtherly, let us put $x^{(k+1)}$ on the left side of the equation, and reach the final iteration formula:

$$x^{(k+1)} = \underbrace{(I - w\tilde L)^{-1}((1-w)I + w\tilde U)}_{N_w}x^{(k)} + \underbrace{w(I-w\tilde L)^{-1}b}_{y_w}$$
 
where  $\tilde{L}=-D^{-1}L,\tilde{U}=-D^{-1}U, y = D^{-1}b$, $w$ is some weight constant.

Algorithm description is similar to those above, only $N$ and $y$ are different. (copied from Wikipedia)

```text
Inputs: A, b, omega
Output: p

Choose an initial guess p to the solution
repeat until convergence
    for i from 1 until n do
        set s to 0
        for j from 1 until n do
            if j != i then
                set s to s + aij pj
            end if
        end (j-loop)
        set pi to (1 − w)pi + w(bi − s) / aii
    end (i-loop)
    check if convergence is reached
end (repeat)
```

# Results

**Apply to P2,3,4**

In Homework 3, Problem 2,3,4

$$
A= \begin{bmatrix}4&1&-1\\-1&3&1\\2&2&6\end{bmatrix},b=[5,-4,1]^T
$$

Of which the true solution is $x=[1.45,-0.8375, -0.0375]^T$. Using the function we write, we got the following output

```text
[ 1.45   -0.8375 -0.0375] 5.4512211510425905e-09
[ 1.45   -0.8375 -0.0375] 1.8484244690485212e-09
[ 1.45   -0.8375 -0.0375] 3.057784848697519e-09
[ 1.45   -0.8375 -0.0375] 8.75190103670142e-09
22 12 13 11
```
i.e each of the method reaches the right solution. Jordan, Gauss-Seidel, SOR with $w=1.05$, SOR with $w=0.95$ costs 22,12,13 and 11 iterations respectively, indicating our implementation is right.

**Time costs** of each method to reach error less than $10^{-8}$ is shown below:

![TimeCost](times.png)

| n            | 10   | 20   | 50   | 100  | 200  | 500  |
| ------------ | ---- | ---- | ---- | ---- | ---- | ---- |
| Jordan       | 2.9178760e-01 | 1.2899970e-01 | 6.9642000e-02 | 6.1297280e-01 | 6.1794190e-01 |1.2913674e+00|
| Gauss-Seidel | 4.1537040e-01 | 6.0859860e-01 | 8.3784770e-01 | 1.4155799e+00 | 2.3733580e+00 | 5.8183695e+00 |
| SOR $w=1.05$ | 0.54621 | 0.682089 | 0.925342 | 1.6800848 | 2.9810761 | 6.8620464 |
| SOR $w=0.95$ | 4.4975870e-01 |6.2030070e-01|9.1496600e-01| 1.5099740e+00|2.8818061e+00| 6.8098993e+00|

**Table:** Time costs ($\varepsilon=10^{-8}$)

**Iteration counts** of each method to reach error less than $10^{-8}$ is shown below:

![IterCost](iters.png)

| n            | 10   | 20   | 50   | 100  | 200  | 500  |
| ------------ | ---- | ---- | ---- | ---- | ---- | ---- |
| Jordan       | 62   | 26   | 16   | 13   | 11   | 9    |
| Gauss-Seidel | 23   | 16   | 12   | 10   | 9    | 8    |
| SOR $w=1.05$ | 27   | 17   | 13   | 12   | 11   | 10   |
| SOR $w=0.95$ | 22   | 16   | 13   | 11   | 11   | 10   |

**Table:** Iteration counts ($n=500,\varepsilon=10^{-8}$)

Fix $n=500$, the **errors** w.r.t iteration count is shown below:

![Err](500_iter.png)

# Analysis

From the results above, we can notice the following points:

1. The time complexity of each method on linear equations of the same size and same tolerance, is likely to be linear.
2. Among all methods, Jordan method is the fastest, then Gauss-Seidel, then SOR with $w=1.05$ and $w=0.95$.
3. From the iterations graph, we are suprised to see that as the size of the matrix $A$ increases, the iteration needed to solve the equations decreases. But referring to the construction of $A$, it is reasonable because as $n$ increases, $A$ becomes more and more diagonally dominant. Solving an almost diagonal matrix equation is super easy. 
4. From the error graph, we could find that Gauss-Seidel method convergence fastest, and the 2 SOR methods performs nearly the same Jordan method, since when $w=1$, SOR degenerates to Gauss-Seidel, in real application of SOR method, we should test out the best $w$ then do the iteration.

# Collaboration

No collaboration on this project.

# Academic Integrity

On my personal intergrity as a student and member of the UCD community, I have given nor received any unauthorized assitance on this assignment.

# Appendix

In [209]:
import numpy as np
import matplotlib.pyplot as plt
from ttictoc import tic, toc
from collections import defaultdict
from tqdm import tqdm

In [210]:
def jordan(A, b, x0, tol, max_iter, force=False, path=None):
    n = 0
    D = np.diag(A)
    Dinv = np.diag(np.divide(1, D))
    

    N = np.identity(np.shape(A)[0]) - Dinv @ A
    y = Dinv @ b

    if path is not None:
        path.clear()
        path.append(x0.copy())
    x1 = N @ x0 + y
    if path is not None:
        path.append(x1.copy())
    while (force or np.linalg.norm(x1 - x0) > tol) and n < max_iter:
        x0 = x1[:]
        x1 = N @ x0 + y
        if path is not None:
            path.append(x1.copy())
        if np.linalg.norm(x1) > 1e8:
            return None, None
        n = n + 1

    return x1, n   
    

In [211]:
def gauss_seidel_step(A, b, x, xp=None):
    if xp is None:
        xp = np.array([] * len(x))

    n = np.shape(A)[0]
    for i in range(n):       
        xp[i] = (b[i] - np.dot(A[i, :i], xp[:i]) -
                    np.dot(A[i, (i+1):], x[(i+1):])) / A[i, i]
    return xp
        
def gauss_seidel(A, b, x0, tol, max_iter, force=False, path=None):
    if path is not None:
        path.clear()
        path.append(x0.copy())
    n = 0
    x1 = np.zeros_like(x0)
    while n == 0 or (force or np.linalg.norm(x0 - x1) > tol) and n < max_iter:
        if n != 0:
            x0 = x1.copy()
        x1 = gauss_seidel_step(A, b, x0, x1)
        if path is not None:
            path.append(x1.copy())
        if np.linalg.norm(x1) > 1e8:
            return None, None
        n = n + 1
    return x1, n


In [212]:
def SOR(A, b, w, x0, tol, max_iter, force=False, path=None):
    if path is not None:
        path.clear()
        path.append(x0.copy())
    n = 0
    x1 = np.zeros_like(x0)
    while n == 0 or (force or np.linalg.norm(x0 - x1) > tol) and n < max_iter:
        if n != 0:
            x0 = x1.copy()
        #for i in range(A.shape[0]):
        temp = np.zeros_like(x0)
        temp = gauss_seidel_step(A, b, x0, temp)
        x1 = (1 - w) * x1 + w * temp
        if path is not None:
            path.append(x1.copy())
        if np.linalg.norm(x1) > 1e8:
            return None, None
        n = n + 1
    return x1, n

#def SOR(A, b, w, x0, tol, max_iter, force=False, path=None):
#    if path is not None:
#        path.clear()
#        path.append(x0[:])
#    L, U, D, sz = np.tril(A), np.triu(A), np.diag(A), np.shape(A)[0]
#    np.fill_diagonal(L, 0)
#    np.fill_diagonal(U, 0)
#    Dinv = np.diag(1 / D)
#
#    Lt, Ut, y = -Dinv @ L, -Dinv @ U, Dinv @ b
#    Iinv = np.linalg.inv(np.identity(sz) - w * Lt)
#    N = Iinv @ ((1 - w) * np.identity(sz) + w * Ut)
#    y = w * Iinv @ y
#
#    x1 = N @ x0 + y
#    n = 0
#    if path is not None:
#        path.append(x1.copy())
#    while (force or np.linalg.norm(x1 - x0) > tol) and n < max_iter:
#        x0 = x1[:]
#        x1 = N @ x0 + y
#        if path is not None:
#            path.append(x1.copy())
#        if np.linalg.norm(x1) > 1e8:
#            return None, None
#        n = n + 1   
#
#    return x1, n

In [213]:
def get_A(n):
    return np.identity(n) * n / 2. + np.random.randn(n * n).reshape((n, n))

ns = np.array([10, 25, 50, 100, 200, 500])
w1, w2 = 1.05, 0.95

legends = {'jordan': 'Jordan', 'gs': 'Gauss Seidel', 'sor1': 'SOR w = 1.05', 'sor2': 'SOR w = 0.95'}

In [214]:
# Problem 2
A = np.array([[4,1,-1],[-1,3,1],[2,2,6]])
b = np.array([5,-4,1])

x1, n1 = jordan(A, b, np.zeros(3), 1e-8, 1000)
x2, n2 = gauss_seidel(A, b, np.zeros(3), 1e-8, 1000)
x3, n3 = SOR(A, b, 1.05, np.zeros(3), 1e-8, 1000)
x4, n4 = SOR(A, b, 0.95, np.zeros(3), 1e-8, 1000)

for x in [x1, x2, x3, x4]:
    print(x, np.linalg.norm(A @ x - b))
print(n1, n2, n3, n4)

[ 1.45   -0.8375 -0.0375] 5.4512211510425905e-09
[ 1.45   -0.8375 -0.0375] 1.8484244690485212e-09
[ 1.45   -0.8375 -0.0375] 3.057784848697519e-09
[ 1.45   -0.8375 -0.0375] 8.75190103670142e-09
22 12 13 11


In [215]:
# 500 Iterations

A = get_A(500)
b = np.random.randn(500)

In [216]:
path = []

errs = {}

x_jordan = jordan(A, b, np.ones(500), 1e-8, 25, True, path)
errs['jordan'] = [np.linalg.norm(A @ s - b, 2) for s in path]

x_gs = gauss_seidel(A, b, np.ones(500), 1e-8, 25, True, path)
errs['gs'] = [np.linalg.norm(A @ s - b, 2) for s in path]

x_sor1 = SOR(A, b, 1.05, np.ones(500), 1e-8, 25, True, path)
errs['sor1'] = [np.linalg.norm(A @ s - b, 2) for s in path]

x_sor2 = SOR(A, b, 0.95, np.ones(500), 1e-8, 25, True, path)
errs['sor2'] = [np.linalg.norm(A @ s - b, 2) for s in path]

In [ ]:
fig = plt.figure()
for k, v in errs.items():
    plt.plot(v, label=legends[k])
plt.legend()
plt.yscale('log')
plt.xlabel("Iteration\n Figure: Descreases of 2-norm errors as we iterate more")
plt.ylabel("Error")
plt.title("Errors as iterating")
fig.savefig("500_iter.png", bbox_inches='tight', dpi=300)

In [ ]:
# iterations

iters = defaultdict(dict)
times = defaultdict(dict)

N = 300
for n in ns:
    print(n)
    A = get_A(n)
    b = np.random.randn(n)

    tic()
    for _ in tqdm(range(N)):
        _, iters['jordan'][n] = jordan(A, b, np.ones(n), 1e-8, 1000)
    t = toc()
    times['jordan'][n] = t

    tic()
    for _ in tqdm(range(N)):
        _, iters['gs'][n] = gauss_seidel(A, b, np.ones(n), 1e-8, 1000)
    t = toc()
    times['gs'][n] = t

    tic()
    for _ in tqdm(range(N)):
        _, iters['sor1'][n] = SOR(A, b, 1.05, np.ones(n), 1e-8, 1000)
    t = toc()
    times['sor1'][n] = t

    tic()
    for _ in tqdm(range(N)):
        _, iters['sor2'][n] = SOR(A, b, 0.95, np.ones(n), 1e-8, 1000)
    t = toc()
    times['sor2'][n] = t



In [ ]:
fig2 = plt.figure()
for k, v in times.items():
    data = np.array([[_k, _v] for _k, _v in v.items()])
    print(data)
    plt.plot(data[:, 0], data[:, 1], label=legends[k])
plt.legend()
#plt.yscale('log')
plt.xlabel("Size\n Figure: Time consumption as size of matrix increases")
plt.ylabel("Time (300 turns)")
plt.title("Time consumption as size of matrix increases")
fig2.savefig("times.png", bbox_inches='tight', dpi=300)

In [ ]:
fig3 = plt.figure()
for k, v in iters.items():
    data = np.array([[_k, _v] for _k, _v in v.items()])
    print(data)
    plt.plot(data[:, 0], data[:, 1], label=legends[k])
plt.legend()
plt.yscale('log')
plt.xlabel("Dimension\n Figure: Time consumption as size of matrix increases")
plt.ylabel("Iteration (300 turns)")
plt.title("Time consumption as size of matrix increases")
fig3.savefig("iters.png", bbox_inches='tight', dpi=300)